## 🔬 5. Fase MODIFY: Diagnóstico e Tratamento de Outliers Fotométricos

Este notebook calcula métricas de cor, exposição, textura e nitidez diretamente dos arquivos RGB, pois a ABT disponível contém metadados de imagem, não features fotométricas. Compara IQR e Z-score robusto (MAD), exibe as imagens extremas para inspeção e aplica winsorização nos percentis 1 e 99, preservando os registros. Os limites são estimados na partição de treino quando disponível.

**Pré-requisito:** as imagens precisam estar acessíveis em `datasets/` ou nos caminhos de `relative_path`. Sem arquivos legíveis, o notebook informa a cobertura e não salva uma ABT tratada incompleta.

In [2]:
# 5.1 Diagnóstico e tratamento de outliers fotométricos
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image, ImageOps
from IPython.display import display, Markdown

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
abt_path = project_root / 'data' / 'processed' / 'abt_sanidade_vegetal.csv'
output_dir = project_root / 'data' / 'processed'
figure_dir = project_root / 'docs' / 'figures'

metadata = pd.read_csv(abt_path)


def resolve_image_path(row):
    """Resolve caminhos relativos e absolutos registrados em diferentes SOs."""
    filepath = str(row.get('filepath', '')).strip()
    relative_path = str(row.get('relative_path', '')).strip().replace('\\', '/')
    candidates = []
    if filepath:
        candidates.append(Path(filepath))
    if relative_path and relative_path.lower() != 'nan':
        relative = Path(relative_path)
        candidates.extend([
            project_root / relative,
            project_root / 'datasets' / relative,
            project_root / 'data' / 'raw' / relative,
        ])
    return next((path for path in candidates if path.is_file()), None)


def extract_image_metrics(image_path):
    """Calcula métricas globais de cor, exposição, textura e nitidez da imagem."""
    with Image.open(image_path) as source:
        rgb = np.asarray(ImageOps.fit(source.convert('RGB'), (256, 256)), dtype=np.float32)
    red, green, blue = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    hsv = np.asarray(Image.fromarray(rgb.astype(np.uint8)).convert('HSV'), dtype=np.float32)
    hue, saturation, value = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    valid_hue = (saturation > 20) & (value > 20)

    gray = (0.299 * red + 0.587 * green + 0.114 * blue) / 255.0
    quantized = np.minimum((gray * 8).astype(np.uint8), 7)
    left, right = quantized[:, :-1].astype(np.float32), quantized[:, 1:].astype(np.float32)
    differences = np.abs(left - right)
    laplacian = (
        -4 * gray[1:-1, 1:-1]
        + gray[:-2, 1:-1] + gray[2:, 1:-1]
        + gray[1:-1, :-2] + gray[1:-1, 2:]
    )

    return {
        'mean_hue': float(hue[valid_hue].mean() * 360.0 / 255.0) if valid_hue.any() else np.nan,
        'std_saturation': float((saturation / 255.0).std()),
        'exg_index': float((2 * green - red - blue).mean()),
        'exr_index': float((1.4 * red - green).mean()),
        'glcm_contrast': float(np.square(differences).mean()),
        'glcm_homogeneity': float((1.0 / (1.0 + differences)).mean()),
        'laplacian_var': float(laplacian.var()),
        'overexposed_fraction': float((value >= 250).mean()),
        'shadow_fraction': float((value <= 25).mean()),
    }


records = []
missing_images = []
for row_index, row in metadata.iterrows():
    image_path = resolve_image_path(row)
    if image_path is None:
        missing_images.append(row_index)
        continue
    try:
        records.append({
            'row_index': row_index,
            'resolved_image_path': str(image_path),
            **extract_image_metrics(image_path),
        })
    except Exception as error:
        missing_images.append(row_index)
        print(f"[AVISO] Falha ao ler imagem da linha {row_index}: {error}")

print(f"Imagens processadas: {len(records):,}/{len(metadata):,}; indisponíveis/ilegíveis: {len(missing_images):,}.")

if not records:
    display(Markdown(
        '**Execução bloqueada:** nenhuma imagem foi localizada. A ABT atual contém apenas metadados; '
        'monte o dataset no diretório `datasets/` ou atualize `relative_path` antes de calcular '
        'estatísticas. Nenhuma ABT tratada foi salva.'
    ))
else:
    metrics = pd.DataFrame(records).set_index('row_index')
    analyzed = metadata.loc[metrics.index].copy()
    analyzed = analyzed.join(metrics)
    feature_columns = [
        'mean_hue', 'std_saturation', 'exg_index', 'exr_index', 'glcm_contrast',
        'glcm_homogeneity', 'laplacian_var', 'overexposed_fraction', 'shadow_fraction',
    ]

    train_mask = analyzed.get('split_partition', pd.Series(index=analyzed.index, dtype='object'))
    train_mask = train_mask.astype(str).str.lower().isin(['train', 'training'])
    if train_mask.any():
        reference = analyzed.loc[train_mask]
        reference_description = 'partição train'
    else:
        reference = analyzed
        reference_description = 'todas as imagens disponíveis (split train não identificado)'

    audit_rows = []
    summary_rows = []
    winsor_bounds = {}
    for feature in feature_columns:
        reference_values = reference[feature].dropna()
        values = analyzed[feature]
        if reference_values.empty:
            continue
        q1, q3 = reference_values.quantile([0.25, 0.75])
        iqr = q3 - q1
        lower_iqr, upper_iqr = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        median = reference_values.median()
        mad = (reference_values - median).abs().median()
        delta = values - median
        if mad > 0:
            robust_z = 0.6745 * delta / mad
        else:
            robust_z = pd.Series(np.where(delta == 0, 0.0, np.sign(delta) * np.inf), index=values.index)
        iqr_flag = (values < lower_iqr) | (values > upper_iqr)
        robust_flag = robust_z.abs() > 3.5
        lower_clip, upper_clip = reference_values.quantile([0.01, 0.99])
        winsor_bounds[feature] = (lower_clip, upper_clip)

        summary_rows.append({
            'feature': feature,
            'n': int(values.notna().sum()),
            'iqr_outliers': int(iqr_flag.sum()),
            'robust_z_outliers': int(robust_flag.sum()),
            'overlap_outliers': int((iqr_flag & robust_flag).sum()),
            'median_reference': median,
            'mad_reference': mad,
            'iqr_lower': lower_iqr,
            'iqr_upper': upper_iqr,
            'winsor_lower_p01': lower_clip,
            'winsor_upper_p99': upper_clip,
        })
        for index in values.index:
            audit_rows.append({
                'sample_id': analyzed.at[index, 'sample_id'] if 'sample_id' in analyzed else index,
                'filepath': analyzed.at[index, 'filepath'] if 'filepath' in analyzed else '',
                'resolved_image_path': analyzed.at[index, 'resolved_image_path'],
                'class_label': analyzed.at[index, 'class_label'] if 'class_label' in analyzed else '',
                'feature': feature,
                'value': values.at[index],
                'iqr_outlier': bool(iqr_flag.at[index]),
                'robust_z': robust_z.at[index],
                'robust_z_outlier': bool(robust_flag.at[index]),
                'is_outlier': bool(iqr_flag.at[index] or robust_flag.at[index]),
            })

    summary = pd.DataFrame(summary_rows)
    audit = pd.DataFrame(audit_rows)
    analyzed_treated = analyzed.copy()
    for feature, (lower, upper) in winsor_bounds.items():
        analyzed_treated[feature] = analyzed_treated[feature].clip(lower=lower, upper=upper)

    output_dir.mkdir(parents=True, exist_ok=True)
    figure_dir.mkdir(parents=True, exist_ok=True)
    summary.to_csv(output_dir / 'relatorio_outliers_fotometricos.csv', index=False)
    audit.to_csv(output_dir / 'auditoria_outliers_fotometricos.csv', index=False)

    columns = 2
    rows = int(np.ceil(len(winsor_bounds) / columns))
    figure = make_subplots(
        rows=rows, cols=columns,
        subplot_titles=list(winsor_bounds),
        vertical_spacing=min(0.12, 0.6 / rows),
    )
    for position, feature in enumerate(winsor_bounds):
        row, column = position // columns + 1, position % columns + 1
        figure.add_trace(go.Box(y=analyzed[feature], name='Original', marker_color='#d95f02', boxpoints='outliers'), row=row, col=column)
        figure.add_trace(go.Box(y=analyzed_treated[feature], name='Winsorizado', marker_color='#1b9e77', boxpoints='outliers'), row=row, col=column)
    figure.update_layout(
        title='Distribuições das features fotométricas antes e depois da winsorização',
        template='plotly_white', height=max(420, 300 * rows),
        boxmode='group', showlegend=True,
    )
    figure.write_html(figure_dir / 'diagnostico_outliers_fotometricos.html', include_plotlyjs='cdn')
    figure.show()

    outlier_audit = audit[audit['is_outlier']].copy()
    if not outlier_audit.empty:
        visual_samples = (
            outlier_audit.assign(abs_robust_z=outlier_audit['robust_z'].abs())
            .sort_values('abs_robust_z', ascending=False)
            .drop_duplicates('sample_id')
            .head(12)
        )
        display(Markdown('### Inspeção visual dos registros extremos'))
        for _, sample in visual_samples.iterrows():
            print(f"{sample['class_label']} | {sample['feature']}={sample['value']:.4g} | {sample['resolved_image_path']}")
            with Image.open(sample['resolved_image_path']) as image:
                display(ImageOps.contain(image.convert('RGB'), (480, 320)))

    coverage_note = (
        'Cobertura completa.' if not missing_images
        else f"Cobertura parcial: {len(missing_images)} de {len(metadata)} imagens não foram lidas; não usar o CSV tratado como base completa."
    )
    top_findings = summary.sort_values('iqr_outliers', ascending=False).head(5)
    findings_table = (
        '| Feature | Outliers IQR | Outliers Z robusto | Em comum |\n'
        '| --- | ---: | ---: | ---: |\n'
        + '\n'.join(
            f"| {item.feature} | {item.iqr_outliers} | {item.robust_z_outliers} | {item.overlap_outliers} |"
            for item in top_findings.itertuples(index=False)
        )
    )
    report = f'''# Relatório estatístico: outliers fotométricos

## Escopo e cobertura

- Amostras na ABT: {len(metadata)}.
- Imagens analisadas: {len(analyzed)}.
- Referência estatística: {reference_description}.
- {coverage_note}

## Método

Foram calculados matiz médio, dispersão de saturação, índices ExG/ExR, contraste e homogeneidade de textura, variância do Laplaciano e proporções de pixels superexpostos/sombreados. O IQR sinaliza valores fora de Q1 - 1,5×IQR e Q3 + 1,5×IQR. O Z-score robusto usa 0,6745×(x - mediana)/MAD, com limiar absoluto 3,5; quando MAD=0, valores diferentes da mediana são sinalizados como extremos. Os limites de winsorização são os percentis 1 e 99 da referência estatística.

## Resultados por feature

{findings_table}

A tabela completa, com limites, MAD e contagens, está em `data/processed/relatorio_outliers_fotometricos.csv`. A auditoria por amostra-feature está em `data/processed/auditoria_outliers_fotometricos.csv`. Os boxplots interativos estão em `docs/figures/diagnostico_outliers_fotometricos.html`.

## Inspeção visual e decisão de tratamento

Os registros sinalizados por qualquer um dos métodos são apresentados no notebook para inspeção visual. A winsorização preserva as amostras e limita a influência de artefatos de captura sobre a escala das features; a remoção não foi adotada porque um extremo pode representar variação legítima de doença, e não erro fotográfico. A inspeção da imagem deve confirmar superexposição, sombra, reflexo ou falha de foco antes de qualquer exclusão manual.

A winsorização deve ser ajustada somente na partição de treino e aplicada sem reajuste às demais partições, evitando vazamento de informação. O arquivo tratado completo só é salvo quando todas as imagens da ABT forem legíveis.
'''
    (project_root / 'docs' / 'relatorio_outliers_fotometricos.md').write_text(report, encoding='utf-8')

    if not missing_images:
        analyzed_treated.to_csv(output_dir / 'abt_sanidade_vegetal_treated.csv', index=False)
        print('[OK] ABT tratada completa salva em data/processed/abt_sanidade_vegetal_treated.csv')
    else:
        print('[AVISO] ABT tratada não foi sobrescrita porque a cobertura de imagens é incompleta.')
    print('[OK] Relatório estatístico, auditoria e boxplots foram gerados.')

Imagens processadas: 0/6,571; indisponíveis/ilegíveis: 6,571.


**Execução bloqueada:** nenhuma imagem foi localizada. A ABT atual contém apenas metadados; monte o dataset no diretório `datasets/` ou atualize `relative_path` antes de calcular estatísticas. Nenhuma ABT tratada foi salva.